# 01 · Sources, retrieval and validation (Classes 4–6)

**Client question:** *"Late deliveries are increasing and customers say our ETA is unreliable. Figure out what is happening before we invest in an AI delay-prediction system."*

This notebook is the investigation that led to the pipeline. It reuses the pipeline's own modules (`pipeline/extract.py`, `pipeline/validate.py`), so what you see here is exactly what runs every day.

| Section | Class | Question |
|---|---|---|
| 1 | 4 | Where does the truth for each business question live? |
| 2 | 5 | Did we retrieve **all** of it, from 3 retrieval modes, and keep the raw inputs? |
| 3 | 5 | Does the client's "56% late" claim hold up? |
| 4 | 6 | What is wrong with the data, and does it matter for the decision? |
| 5 | 6 | Business rules as code: rule → check → output → handling |

In [1]:
import sys, json, logging, tempfile, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s | %(message)s", force=True)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
log = logging.getLogger("notebook")

from pipeline.config import load_config
cfg = load_config(root=ROOT)

## 1 · Source map (Class 4)

Start from the questions, not the tables. The full map with owners/freshness is in [`docs/source_map.md`](../docs/source_map.md).

| Business question | Information needed | System of record | Mode |
|---|---|---|---|
| How many orders are late? | promised ETA, actual delivery, final status | Order platform DB (`orders`) | SQL |
| Where in the workflow is time lost? | assigned, picked-up, delivered times | Dispatch service + order DB | API + SQL |
| Did the driver actually move? | GPS pings between pickup and delivery | Driver app (`driver_events.json`) | nested JSON |
| Was the restaurant ready? | restaurant status + last update | Restaurant status export | CSV |
| Did Ops/Support intervene, and when? | intervention type, time, initiator | Intervention log | CSV |
| What did customers experience? | tickets, ETA views, support opens | Support desk + app analytics | CSV |

## 2 · Retrieval with completeness proof (Class 5)

A fresh mock Dispatch API is started for this run. Watch the log: page 3 returns **HTTP 500** and page 5 **HTTP 429** on first hit; both are retried, nothing is dropped.

In [2]:
from pipeline.extract import extract_all
from pipeline.mock_api import mock_dispatch_api

raw_dir = Path(tempfile.mkdtemp()) / "bronze_notebook"
with mock_dispatch_api(ROOT, log) as api_url:
    sources = extract_all(cfg, "2026-08-01", "2026-08-29", api_url, raw_dir, log)

manifest = pd.DataFrame({k: r.manifest() for k, r in sources.items()}).T
manifest[["mode", "source", "rows", "expected_rows", "complete", "completeness_check"]]

INFO    | Mock Dispatch API healthy | url=http://127.0.0.1:56203 records=1600


INFO    | SQL orders       rows=1603 COUNT(*)=1603 complete=True


INFO    | SQL order_keys   rows=1603 COUNT(*)=1603 complete=True


INFO    | SQL customers    rows=900 COUNT(*)=900 complete=True


INFO    | SQL drivers      rows=120 COUNT(*)=120 complete=True


INFO    | SQL restaurants  rows=60 COUNT(*)=60 complete=True


... (16 outputs left out)

INFO    | Dispatch API page=8 rows=200 cumulative=1600/1600


INFO    | API  dispatch     pages=8 rows=1600 total_records=1600 complete=True


INFO    | Mock Dispatch API stopped


,mode,source,rows,expected_rows,complete,completeness_check
orders,SQL,flasheats.db:orders,1603,1603,True,len(result) == SELECT COUNT(*) with same filter
order_keys,SQL,flasheats.db:order_keys,1603,1603,True,len(result) == SELECT COUNT(*) with same filter
customers,SQL,flasheats.db:customers,900,900,True,len(result) == SELECT COUNT(*) with same filter
drivers,SQL,flasheats.db:drivers,120,120,True,len(result) == SELECT COUNT(*) with same filter
restaurants,SQL,flasheats.db:restaurants,60,60,True,len(result) == SELECT COUNT(*) with same filter
interventions,CSV,order_interventions.csv,430,430,True,parsed rows == csv records in file
support_tickets,CSV,support_tickets.csv,202,202,True,parsed rows == csv records in file
restaurant_status,CSV,restaurant_status.csv,502,502,True,parsed rows == csv records in file
app_actions,CSV,customer_app_actions.csv,2365,2365,True,parsed rows == csv records in file
order_events,CSV,order_events.csv,4955,4955,True,parsed rows == csv records in file


In [3]:
print("Raw inputs preserved:")
for p in sorted(raw_dir.rglob("*")):
    if p.is_file(): print("  ", p.relative_to(raw_dir), f"{p.stat().st_size/1024:,.0f} KB")

Raw inputs preserved:
   customer_app_actions.csv 151 KB
   dispatch_api/page_001.json 67 KB
   dispatch_api/page_002.json 67 KB
   dispatch_api/page_003.json 67 KB
   dispatch_api/page_004.json 67 KB
   dispatch_api/page_005.json 67 KB
   dispatch_api/page_006.json 67 KB
   dispatch_api/page_007.json 67 KB
   dispatch_api/page_008.json 67 KB
   driver_events.json 1,506 KB
   manifest.json 4 KB
   order_events.csv 396 KB
   order_interventions.csv 34 KB
   order_outcomes.csv 69 KB
   restaurant_status.csv 20 KB
   sql_customers.csv 5 KB
   sql_drivers.csv 2 KB
   sql_order_keys.csv 42 KB
   sql_orders.csv 244 KB
   sql_restaurants.csv 3 KB
   support_tickets.csv 20 KB


**How we know retrieval is complete**
* SQL: `len(result) == SELECT COUNT(*)` with the same `WHERE` clause.
* CSV: parsed rows == CSV records counted independently with the `csv` module (+ sha256 in the manifest).
* Nested JSON: flattened rows == sum of `len(events)` across all 120 drivers.
* API: sum of page rows == `total_records` returned by the API, and every `order_id` is unique.

Any mismatch raises `ExtractError` and the run publishes nothing.

## 3 · Testing the client's claim: "Late Delivery Rate is 56%" (Class 5)

Four stakeholders define *late* differently (`client_metric_definitions.json`). Before calculating, state the definition and the denominator.

In [4]:
print(json.dumps(json.loads((ROOT / "source_data/files/client_metric_definitions.json").read_text()), indent=2))

{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


In [5]:
o = sources["orders"].df.drop_duplicates("order_id").copy()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    o[c] = pd.to_datetime(o[c], format="mixed", errors="coerce")
o["status"] = o["final_status"].str.strip().str.lower()
o["delay_min"] = (o["actual_delivery_at"] - o["promised_eta"]).dt.total_seconds() / 60

funnel = pd.Series({
    "raw rows": len(sources["orders"].df),
    "unique orders": len(o),
    "cancelled (excluded, Finance)": int((o.status == "cancelled").sum()),
    "delivered": int((o.status == "delivered").sum()),
    "delivered but no delivery time (unmeasurable)": int(((o.status == "delivered") & o.actual_delivery_at.isna()).sum()),
    "measurable deliveries (denominator)": int(((o.status == "delivered") & o.actual_delivery_at.notna()).sum()),
    "late (delay > 0 min)": int((o.delay_min > 0).sum()),
})
funnel.to_frame("orders")

,orders
raw rows,1603
unique orders,1600
"cancelled (excluded, Finance)",68
delivered,1532
delivered but no delivery time (unmeasurable),37
measurable deliveries (denominator),1495
late (delay > 0 min),843


In [6]:
print(f"Late rate = {funnel['late (delay > 0 min)']} / {funnel['measurable deliveries (denominator)']} = "
      f"{funnel['late (delay > 0 min)'] / funnel['measurable deliveries (denominator)']:.1%}")

Late rate = 843 / 1495 = 56.4%


**Finding:** the 56% claim is reproducible, but *only* with one specific denominator: delivered orders that have a delivery timestamp. The same data gives 23% under the Support Lead's >10 min rule and 52.7% if cancelled orders are wrongly left in. The pipeline reports all of them side by side (`late_definition_sensitivity.csv`). The 56% is not wrong, but without its definition it isn't safe to quote.

## 4 · Profiling (Class 6)

Profiling tells us where to investigate; it does not tell us what to delete.

In [7]:
raw = sources["orders"].df
profile = pd.DataFrame({"dtype": raw.dtypes.astype(str), "nulls": raw.isna().sum(), "distinct": raw.nunique()})
profile

,dtype,nulls,distinct
order_id,object,0,1600
customer_id,object,0,727
restaurant_id,object,3,60
driver_id,object,3,120
city,object,0,1
created_at,object,0,1527
promised_eta,object,0,1592
pickup_at,object,0,1600
actual_delivery_at,object,105,1495
final_status,object,0,3


In [8]:
for col in ["final_status", "traffic_bucket", "weather_bucket"]:
    print(col, raw[col].value_counts(dropna=False).to_dict())
print("restaurant status:", sources["restaurant_status"].df["status"].value_counts(dropna=False).to_dict())
print("ticket category:  ", sources["support_tickets"].df["category"].value_counts(dropna=False).to_dict())

final_status {'delivered': 1530, 'cancelled': 68, 'Delivered': 5}
traffic_bucket {'medium': 635, 'high': 448, 'low': 381, 'severe': 136, 'HIGH': 3}
weather_bucket {'clear': 1276, 'rain': 249, 'heavy_rain': 78}
restaurant status: {'preparing': 171, 'handed_off': 168, 'ready': 157, 'ready ': 2, 'READY': 1, 'Ready': 1, 'handoff': 1, 'unknown': 1}
ticket category:   {'late_delivery': 38, 'eta_changed': 37, 'restaurant_delay': 37, 'ready_but_waiting': 33, 'status_mismatch': 29, 'driver_not_moving': 25, 'Late Delivery': 1, 'late_delivery ': 1, 'ETA issue': 1}


### 4a · The duplicates are not identical: which copy we keep matters for traffic, not for the KPI

In [9]:
dups = raw[raw.order_id.duplicated(keep=False)].sort_values("order_id")
dups[["order_id", "created_at", "promised_eta", "actual_delivery_at", "final_status", "traffic_bucket"]]

,order_id,created_at,promised_eta,actual_delivery_at,final_status,traffic_bucket
119,O00120,2026-08-13T18:15:00,2026-08-13T19:40:24,2026-08-13T19:34:08.431508,delivered,severe
1600,O00120,2026-08-13T18:15:00,2026-08-13T19:40:24,2026-08-13T19:34:08.431508,delivered,medium
722,O00723,2026-08-05T21:04:00,2026-08-05T22:19:36,2026-08-05T22:04:30.928322,delivered,medium
1601,O00723,2026-08-05T21:04:00,2026-08-05T22:19:36,2026-08-05T22:04:30.928322,delivered,high
1301,O01302,2026-08-26T21:07:00,2026-08-26T22:15:50.273095,2026-08-26T22:20:27.428561,delivered,medium
1602,O01302,2026-08-26T21:07:00,2026-08-26T22:15:50.273095,2026-08-26T22:20:27.428561,delivered,high


### 4b · Lifecycle chronology

In [10]:
d = sources["dispatch"].df.set_index("order_id")
o["assigned_at"] = pd.to_datetime(o.order_id.map(d["assigned_at"]), format="mixed")
chronology = pd.Series({
    "delivered before pickup": int((o.actual_delivery_at < o.pickup_at).sum()),
    "promised ETA before order created": int((o.promised_eta < o.created_at).sum()),
    "picked up before driver assigned (delivered orders)": int(((o.pickup_at < o.assigned_at) & (o.status == "delivered")).sum()),
    "cancelled orders WITH a pickup_at": int(((o.status == "cancelled") & o.pickup_at.notna()).sum()),
})
chronology.to_frame("orders")

,orders
delivered before pickup,5
promised ETA before order created,4
picked up before driver assigned (delivered orders),14
cancelled orders WITH a pickup_at,68


### 4c · Cross-source agreement: do systems tell the same story?

In [11]:
de = sources["driver_events"].df.copy()
de["timestamp"] = pd.to_datetime(de.timestamp, format="mixed")
app_deliv = de[de.type == "delivered"].groupby("order_id").timestamp.min()
both = o[o.actual_delivery_at.notna()].copy()
both["app_delivered_at"] = both.order_id.map(app_deliv)
unmeasurable = o[(o.status == "delivered") & o.actual_delivery_at.isna()]
print("Driver app vs order DB delivery time, max abs diff (s):",
      (both.actual_delivery_at - both.app_delivered_at).dt.total_seconds().abs().max())
print("Unmeasurable orders that DO have a driver-app delivery event:",
      unmeasurable.order_id.isin(app_deliv.index).sum(), "of", len(unmeasurable))

ev = sources["order_events"].df
iv = sources["interventions"].df
a, b = set(iv.order_id), set(ev.loc[ev.source_system == "interventions", "order_id"])
print(f"\nIntervention log orders={len(a)} | order_events intervention orders={len(b)} | overlap={len(a & b)}")

Driver app vs order DB delivery time, max abs diff (s): 0.0
Unmeasurable orders that DO have a driver-app delivery event: 37 of 37

Intervention log orders=430 | order_events intervention orders=260 | overlap=75


**Judgement calls from profiling**
* **Driver app can fill the 37 missing delivery times,** and it agrees with the order DB to the second on every other order. We still **do not back-fill**: the order DB is the system of record, and the gap is an instrumentation bug the client should fix. We report the back-filled rate as a sensitivity (56.6% vs 56.4%). It doesn't change the decision.
* **Two intervention systems disagree:** only 75 orders are in both. We use `order_interventions.csv` (it has initiator + reason) and raise the discrepancy with Ops. Intervention metrics are labelled with that caveat.

## 5 · Business rules as code (Class 6)

Each rule has a plain-English statement, a severity, the affected count, and the **handling**. ERROR stops publication; WARN publishes with the issue disclosed.

In [12]:
from pipeline.validate import validate_sources
from datetime import date
checks = pd.DataFrame([c.to_dict() for c in validate_sources(sources, cfg, date(2026, 8, 29))])
checks[checks.status != "PASS"][["rule_id", "category", "rule", "status", "affected", "handling"]]

,rule_id,category,rule,status,affected,handling
7,S04,structural,order_id is unique (grain = one row per order),WARN,3,"Keep first occurrence (agreed rule). Timestamps identical, so KPI unaffected; traffic_..."
9,S06,structural,final_status uses canonical spelling,INFO,5,Normalised to lower-case in clean step
11,S06,structural,traffic_bucket uses canonical spelling,INFO,3,Normalised to lower-case in clean step
15,S08,structural,orders.driver_id is populated,WARN,3,Kept in KPI; excluded from per-driver/restaurant breakdowns
16,S08,structural,orders.restaurant_id is populated,WARN,3,Kept in KPI; excluded from per-driver/restaurant breakdowns
20,L02,lifecycle,Delivered orders have actual_delivery_at,WARN,37,Excluded from KPI denominator and reported as 'unmeasurable'; NOT back-filled (driver-...
21,L03,lifecycle,Delivery happens after pickup,WARN,5,Kept in KPI (delay uses promised vs delivered only); excluded from pickup->delivered s...
22,L04,lifecycle,Promised ETA is after order creation,WARN,4,Kept in KPI (max impact 0.3 pp) and listed as assumption; needs order-platform owner t...
23,L05,lifecycle,Cancelled orders have no pickup timestamp,INFO,68,Cancelled orders excluded from all stage metrics
24,L06,lifecycle,Pickup happens after dispatch assignment,WARN,14,Excluded from assigned->pickup stage metric


In [13]:
checks.groupby(["category", "status"]).size().unstack(fill_value=0)

status,INFO,PASS,WARN
category,,,
cross-source,2,13,4
lifecycle,1,3,4
structural,2,12,3


The same rules run in `run_pipeline.py` on every run. Continue with [`02_workflow_model_and_metrics.ipynb`](02_workflow_model_and_metrics.ipynb).